# Historical pilot notebook — sanitized
Downloaded from the completed Colab run on 17 October 2026. The original code cells had execution counts 1 and 2.
Embedded payload and outputs are removed; setup now expects a private extracted archive. This edited snapshot has NOT been rerun. Use ../../pilot.ipynb for new runs.
The exact original and full outputs are preserved privately under runs/pilot-2026-10-17/executed-colab.ipynb.


# Algoverse: actual 50-record exploratory pilot
Both models; B/R/N/R2; both mappings; 800 judgments plus 4 identity calls. Native BF16 GPU required. Source labels/text remain unchanged. Qualification flags remain attached. This is not final evaluation.

In [ ]:
import pathlib, base64, json, subprocess, sys, torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0), flush=True)
assert torch.cuda.is_bf16_supported(including_emulation=False), "This pilot needs native BF16: select L4/A100 or equivalent."
ROOT = pathlib.Path('/content/algoverse-pilot-2026-10-17')
ROOT.mkdir(exist_ok=False)
# SANITIZED: original inline payload removed. Supply the private extracted run archive.
PRIVATE_ARCHIVE = pathlib.Path('/content/private-pilot-archive')
payload = {name: base64.b64encode((PRIVATE_ARCHIVE/name).read_bytes()).decode()
           for name in ['inference.py', 'protocol.py', 'summarize_pilot.py', 'candidate_inputs.jsonl', 'qualification.json']}
for name, data in payload.items():
    (ROOT/name).write_bytes(base64.b64decode(data))
subprocess.run([sys.executable,'-m','pip','install','transformers==5.19.0','mistral-common==1.12.0','accelerate','bitsandbytes','huggingface_hub','sentencepiece'], check=True)
(ROOT/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Pilot inputs and runner ready:', ROOT)


In [ ]:
import pathlib, subprocess, sys, time, json, shutil
ROOT = pathlib.Path('/content/algoverse-pilot-2026-10-17')
revisions={'ministral':'f6fae9795746f63c9be8344932f01275f3c63734','gemma':'707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7'}
start=time.time()
for model,revision in revisions.items():
    cmd=[sys.executable,'-u',str(ROOT/'inference.py'),'--model',model,'--revision',revision,'--compute-dtype','bfloat16','--input',str(ROOT/'candidate_inputs.jsonl'),'--out',str(ROOT/model),'--variants','B','R','N','R2']
    print('STARTING REAL INFERENCE:',model,'400 judgments + 2 identity checks',flush=True)
    with (ROOT/(model+'.log')).open('w') as log:
        proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            log.write(line);log.flush();print(line,end='',flush=True)
        if proc.wait():raise RuntimeError(model+' failed; logs and partial outputs retained')
    print('FINISHED',model,'elapsed seconds',round(time.time()-start),flush=True)
subprocess.run([sys.executable,str(ROOT/'summarize_pilot.py'),str(ROOT)],check=True)
print('PILOT COMPLETE. Exploratory only; source qualification remains pending.',flush=True)
archive=shutil.make_archive(str(ROOT), 'zip', ROOT)
from google.colab import files
files.download(archive)
